<center><a target="_blank" href="https://academy.constructor.org/"><img src=https://lh3.googleusercontent.com/d/1fypIr9T-7ntcsVQFmC2_iMPcsm7h8jXg width="500" style="background:none; border:none; box-shadow:none;" /></a> </center>
<hr />

# <h1 align="center"> Day-3-Inductive-Statistics. Part 3 </h1> </center>

<p style="margin-bottom:1cm;"></p>

_____

<center>Constructor Nexademy, 2026</center>


# In-class Tutorial + Exercise (30 mins)

## Part 3.  Type I or Type II error. Permutations. Power analysis.

## A brief introduction

In this tutorial you will get familiar with elements of **Inductive Statistics**.

<span style="color: red;"> **The exercises are annotated as follows:** </span>

### Exercise
text of the task

In [ ]:
# your code here

Concepts covered

* Type I or Type II error.
* Permutations.
* Power abalysis.

### Exercise. T-test and Type I or Type II error

You work on treatment for diabetes patients. You want to understand if there is a relationship between weight and diabetes. Unfortunately, due to privacy protection laws, you are not able to get the exact weight of diabetes patients. You know, however, that normal distribution approximates well weight. The weight of healthy people follows ~ N(45,5), and the weight of diabetis patients follows ~ N(50, 5). With this information, you can make the conclusion on the null hypothesis. Now, you want also understand how often there is an error in this reasoning.

Draw many (1000) sets of samples with 30 observations each, where:

Both samples are from the same normal distribution X ~ N(50,5);
Each sample is from a different normal distribution, one from X ~ N(50, 5) and one from X ~ N(45, 5).

Run a t-test for each set of samples. Using this simulation, how often is there a type I or type II error?
**Bonus**: What happens if you change the sample sizes or the difference in means? We’ll talk about this “power” to detect effects later today.


In [1]:
# your code here
import numpy as np
import scipy.stats as stats

np.random.seed(42)

n_simulations = 1000
n_obs = 30

In [3]:
type_i_errors = 0
for _ in range(n_simulations):
    s1 = np.random.normal(50, 5, n_obs)
    s2 = np.random.normal(50, 5, n_obs)
    t_stat, p_val = stats.ttest_ind(s1, s2)
    if p_val < 0.05:
        type_i_errors += 1

In [4]:
type_ii_errors = 0

for _ in range(n_simulations):
    s1 = np.random.normal(50, 5, n_obs)
    s2 = np.random.normal(45, 5, n_obs)
    t_stat, p_val = stats.ttest_ind(s1, s2)
    if p_val >= 0.05:
        type_ii_errors += 1

print(f"Type I Error Rate (False Positive): {type_i_errors / n_simulations:.3f}")
print(f"Type II Error Rate (False Negative): {type_ii_errors / n_simulations:.3f}")

Type I Error Rate (False Positive): 0.040
Type II Error Rate (False Negative): 0.035


Increasing the sample size makes the Type II error drop because more data makes the test more powerful and accurate. However, a smaller mean difference makes the Type II error jump because if the groups are too similar, the test becomes blind to the difference.



### Exercise. Permutation tests.

Let’s go back to **plant_growth.csv** dataset. You did a literature review on your plants, and have doubts about your normality assumption.
Use a permutation test to decide if the treatment2 had an effect. In other words, is this grouping of observations into treatment and control groups likely to happen randomly?

Get all possible rearrangements of these observations into the two groups.
For each rearrangement (combination), calculate the difference in group means.
See what range 99% of these group means fall into.
See if the difference in means we observe in the data falls within this 99% range.
Alternatively, to get a p-value, see what the probability of observing a difference of the same or larger magnitude than what we saw in the data

In [5]:
# your code here

import pandas as pd
import numpy as np 


data_path = r"C:\Users\SoRaaN\Desktop\karzanmahmoudi\02_Statistics\day3\plant_growth.csv"
df_plant = pd.read_csv(data_path)
df_plant



,Unnamed: 0,weight,group
0,1,4.17,ctrl
1,2,5.58,ctrl
2,3,5.18,ctrl
3,4,6.11,ctrl
4,5,4.50,ctrl
5,6,4.61,ctrl
6,7,5.17,ctrl
7,8,4.53,ctrl
8,9,5.33,ctrl
9,10,5.14,ctrl


In [6]:
ctrl_weights = df_plant[df_plant['group'] == 'ctrl']['weight'].values
trt2_weights = df_plant[df_plant['group'] == 'trt2']['weight'].values

observed_diff = np.mean(trt2_weights) - np.mean(ctrl_weights)

combined_weights = np.concatenate([ctrl_weights, trt2_weights])

In [8]:
n_iterations = 10000
diffs = []

for _ in range(n_iterations):
    shuffled = np.random.permutation(combined_weights)
    new_ctrl = shuffled[:len(ctrl_weights)]
    new_trt2 = shuffled[len(ctrl_weights):]
    diffs.append(np.mean(new_trt2) - np.mean(new_ctrl))

diffs = np.array(diffs)

In [9]:
lower_limit = np.percentile(diffs, 0.5)
upper_limit = np.percentile(diffs, 99.5)


p_value = np.mean(np.abs(diffs) >= np.abs(observed_diff))

print(f"Observed Difference: {observed_diff:.4f}")
print(f"99% Confidence Range: [{lower_limit:.4f}, {upper_limit:.4f}]")
print(f"P-value: {p_value:.4f}")

Observed Difference: 0.4940
99% Confidence Range: [-0.6080, 0.6180]
P-value: 0.0459


### Exercise. Power abalysis

Due to Covid outbreak, the government wants to introduce the obligation to wear face masks in public spaces. It’s cumbersome and costly, so they want to be sure wearing a mask reduces the disease spread. To verify it, a control study will be conducted. What is the minimum sample size for such study?

Consider the following parameters

d = .2, α = .01, β = 0.1  
d = .2, α = .05, β = 0.1  
d = .3, α = .01, β = 0.2  
d = .3, α = .05, β = 0.2

In [10]:
# your code here
from statsmodels.stats.power import TTestIndPower

In [16]:
analysis = TTestIndPower()

parameters = [
    {'d': 0.2, 'alpha': 0.01, 'beta': 0.1},
    {'d': 0.2, 'alpha': 0.05, 'beta': 0.1},
    {'d': 0.3, 'alpha': 0.01, 'beta': 0.2},
    {'d': 0.3, 'alpha': 0.05, 'beta': 0.2}
]



In [15]:
print("min dample ize for each group:")
for p in parameters:
    power = 1 - p['beta']
    size = analysis.solve_power(effect_size=p['d'], alpha=p['alpha'], power=power, ratio=1.0)
    print(f"For d={p['d']}, alpha={p['alpha']}, beta={p['beta']} -> Sample Size: {round(size)}")

min dample ize for each group:
For d=0.2, alpha=0.01, beta=0.1 -> Sample Size: 746
For d=0.2, alpha=0.05, beta=0.1 -> Sample Size: 526
For d=0.3, alpha=0.01, beta=0.2 -> Sample Size: 261
For d=0.3, alpha=0.05, beta=0.2 -> Sample Size: 175
